# ATPESC Academy Agents Tutorial

This tutorial notebook demonstrates how to create agentic systems. These are systems that LLMs don't just respond to questions, but actively decide what actions to take. The tutorial has four examples. 

1. **Hello World** — a simple agent to get familiar with the basics
2. **Tool Calling** — giving agents the ability to run real computations
3. **Multi-Agent Wordle** — multiple agents collaborating to solve a problem
4. **Remote agents** — deploying an agent remotely on Polaris using Globus Compute

To complete the tutorial you'll need LangChain and Academy. Install them with:

```bash
pip install langchain langchain-openai academy
```

In [ ]:
!pip install globus_compute_sdk langchain_openai langchain academy-py

## Configuration

We connect to a language model via an OpenAI-compatible API. By default this tutorial uses the ANL inference service, but any OpenAI-compatible endpoint works. You can swap in your own `api_key`, `model_name`, and `base_url`.



In [ ]:
from inference_auth_token import get_access_token
api_key = get_access_token()
model_name = 'openai/gpt-oss-120b' 
base_url = 'https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1'

## Example 1: Simple Agent

We start with the simplest possible interaction — invoking the LLM directly with a single question and printing the response.


In [ ]:
from langchain_openai import ChatOpenAI

# Initialize the model
llm = ChatOpenAI(
    model_name=model_name, 
    api_key=api_key, 
    base_url=base_url
)

# Send a message and get a response
response = llm.invoke("What is ANL ATPESC?")

print(response.content)

## Example 2: Tool calling

Large language models are powerful at reasoning, but they can't *execute* code or run computations on their own. **Tool calling** bridges this gap — it lets an LLM decide *which* function to run based on a user's request, then hands off execution to your code.

We define three tools (`compute_primes`, `compute_energy`, `compute_trajectory`) and a prompt that describes when to use each one. The agent receives a random scientific question, picks the appropriate tool, runs it, and returns the answer.

In [ ]:
def compute_primes(n: int = 1000) -> dict:
    """Count prime numbers up to n."""
    def is_prime(x):
        if x < 2:
            return False
        for i in range(2, int(x**0.5) + 1):
            if x % i == 0:
                return False
        return True

    primes = [x for x in range(2, n + 1) if is_prime(x)]
    return {
        "task": "prime_count",
        "n": n,
        "count": len(primes),
        "largest": primes[-1] if primes else None,
        "first_10": primes[:10]
    }

def compute_energy(molecule: str = "H2O") -> dict:
    """Simulate molecular energy calculation."""
    import random
    # Mock responses - real version would use PySCF or similar
    energies = {"H2O": -76.4, "CO2": -188.5, "CH4": -40.5,"N2": -109.5}
    base_energy = energies.get(molecule, -50.0) + random.uniform(-0.1, 0.1)
    return {
        "task": "energy_calculation",
        "molecule": molecule,
        "energy_hartree": base_energy,
        "method": "surrogate_dft",
        "basis": "cc-pVDZ"
    }

def compute_trajectory(n_atoms: int = 27, n_steps: int = 100) -> dict:
    """Simulate MD trajectory."""
    import random
    temperatures = [300 + random.uniform(-20, 20) for _ in range(n_steps)]
    return {
        "task": "md_simulation",
        "n_atoms": n_atoms,
        "n_steps": n_steps,
        "avg_temperature": sum(temperatures) / len(temperatures),
        "final_temperature": temperatures[-1],
    }

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
import json
import random

requests=["How many prime numbers are there below 500?",
        "What is the potential energy of a water molecule",
        "What is the MD trajectory of water molecules"]

llm = ChatOpenAI(
    model=model_name,
    api_key=api_key,
    base_url=base_url,
)

CHOOSER_PROMPT = """You are a task router. Given a user request, choose which computation to run.

Available computations:
1. "primes" - Count prime numbers (for math/number questions)
2. "energy" - Calculate molecular energy (for chemistry/molecule questions)
3. "trajectory" - Run MD simulation (for physics/dynamics/trajectory questions)
"""

tools = [compute_primes, compute_energy, compute_trajectory]
agent = create_agent(
    llm, 
    tools=tools,
    system_prompt=CHOOSER_PROMPT,
)

# Run the agent
request=random.choice(requests)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": request}]},
    stream_mode='updates',
    version="v2"
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

## Example 3: Multi-agent worldle example

In the final example we run multiple instances of a `WordleAgent` collaborating to solve a game of Wordle. Each agent independently uses a ReAct loop to make guesses via the `wordle_feedback` tool, receiving per-letter feedback (G = correct position, Y = wrong position, B = not in word) to narrow down the target word.

What makes this multi-agent is the collaboration: as each agent learns from its guesses, it shares insights with its peers via `share_information`. Agents incorporate these incoming insights into their prompts via a shared "blackboard", so the group collectively converges on the answer faster than any single agent would alone.

This pattern demonstrates how Academy enables distributed agent coordination, and how the same approach could scale to more computationally expensive tasks beyond a word game.

In [ ]:
from __future__ import annotations

import asyncio
import logging
from typing import Any

from academy.agent import action
from academy.agent import Agent
from academy.context import ActionContext
from academy.exchange import LocalExchangeFactory
from academy.exchange.transport import AgentRegistrationT
from academy.handle import Handle
from academy.identifier import AgentId
from academy.logging.recommended import recommended_logging
from academy.manager import Manager
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.middleware import ModelRequest
from langchain.agents.middleware import ModelRetryMiddleware

from wordle import wordle_feedback
from wordle import TARGET_WORD


logger = logging.getLogger(__name__)



class WordleAgent(Agent):
    """A LLM powered agent playing wordle.

    This agent uses a ReAct loop with a tool to make guesses to a wordle
    game, and a tool to collaborate with other agents. This pattern coudld
    be used to build "agent forests". Academy allows there agents to run in
    a distributed environment, for instance to replace the Wordle game with
    a more computationally expensive simulation application.

    Args:
        peers: A list of other agent handles.
    """

    def __init__(
        self,
        peers: list[Handle[WordleAgent]],
        model = 'openai:gpt-5.4-nano',
        api_key = '',
        base_url = ''
    ) -> None:

        self.peers = peers
        self.insight_dict: dict[AgentId[Any], str] = {}
        self.model = model
        self.api_key = api_key
        self.base_url = base_url

    def prompt_with_blackboard(self) -> str:
        """Generate a prompt containing the blackboard.

        As agent receives information from it's peers, method injects the
        information into the prompt used to query the model.

        Args:
            request: Request made to the model.
        """
        insights = '\n'.join(
            f'- Agent {agent_id}: {insight}'
            for agent_id, insight in self.insight_dict.items()
        )
        insight_section = (
            f'\n\nInsights from peer agents:\n{insights}'
            if insights
            else '\n\nNo insights from peer agents yet.'
        )
        return (
            'You are playing Wordle, a word-guessing game. '
            'The target is a 5-letter English word. '
            'The `wordle_feedback` tool has an internal hiddent word '
            'that you are trying to guess.'
            'Use the `wordle_feedback` tool to make guesses. '
            'After each guess you receive per-letter feedback: '
            'G (correct position), Y (wrong position), B (not in word). '
            'Use this feedback to narrow down the answer. '
            'Collaborate with peer agents by sharing what you learn using '
            '`share_information`, and incorporate their insights below '
            'into your reasoning to solve the puzzle faster.'
            'THIS IS A GAME TO PLAY AROUND WITH AGENT COLLABORATION.'
            + insight_section
        )

    @action(context=True)
    async def receive_information(
        self,
        info: str,
        *,
        context: ActionContext,
    ) -> None:
        """Receive information from other agents playing the same game.

        Args:
            info: Information received from other agent.

        """
        self.insight_dict[context.source_id] = info

        insights = '\n'.join(
            f'- Agent {agent_id}: {insight}'
            for agent_id, insight in self.insight_dict.items()
        )
        logger.info(f'\n\nCurrent blackboard state:\n{insights}')

    async def share_information(self, info: str) -> None:
        """Share information with other agents.

        By sharing information with other agents, as a group you can solve
        the puzzle faster. For instance after you conclude the position of
        some letters you should say "My guesses reveal the target has letters
        _ _ x _ y and z appears in one of the remaining positions."

        Args:
            info: The information to share.
        """
        logger.info(f'Sharing info with {len(self.peers)} agents.')
        await asyncio.gather(
            *(peer.receive_information(info) for peer in self.peers),
        )

    async def agent_on_startup(self):
        """Initialize the academy agent state."""

        @dynamic_prompt
        def system_prompt(request: ModelRequest) -> str:
            return self.prompt_with_blackboard()

        llm = ChatOpenAI(
            model=model_name,
            api_key=api_key,
            base_url=base_url,
        )
        
        self.agent = create_agent(
            llm,            
            tools=[self.share_information, wordle_feedback],
            middleware=[
                system_prompt,
                ModelRetryMiddleware(),
                ModelCallLimitMiddleware(thread_limit=8),
            ],
        )

        
    @action
    async def play_wordle(self) -> str:
        result = await self.agent.ainvoke(
            {
                'messages': [
                    {
                        'role': 'user',
                        'content': (
                            'Play wordle by collaborating with other agents '
                            'and guessing words using `wordle_feedback`. '
                            'Return the answer to the wordle puzzle.'
                        ),
                    },
                ],
            },
        )
        return result

In [ ]:
async with await Manager.from_exchange_factory(
        factory=LocalExchangeFactory(),
        log_config=recommended_logging(),
) as manager:
    # Academy allows you to create circular dependencies by
    # creating mailboxes before launching agents
    agent_registrations: list[AgentRegistrationT] = []
    for _ in range(2):
        agent_registrations.append(
            await manager.register_agent(WordleAgent),
        )
    peers = [Handle(reg.agent_id) for reg in agent_registrations]

    # The pre-created mailboxes are then used to launch the agents
    logger.info('Launching agents.')
    hdls: Handle[WordleAgent] = []
    for wordle_agent_reg in agent_registrations:
        hdls.append(
            await manager.launch(
                WordleAgent,
                # The peer handles are used to create the
                # communication graph
                args=(peers, model_name, api_key, base_url),
                registration=wordle_agent_reg,
            ),
        )

    logger.info('Starting agent actions.')
    results = await asyncio.gather(*(hdl.play_wordle() for hdl in hdls))
    for hdl, result in zip(hdls, results, strict=True):
        logger.info(
            f'{hdl.agent_id} returned {result["messages"][-1].content}',
        )

    logger.info(f'True target word: {TARGET_WORD}')

## 4. Deploying Agents remotely

We use Academy and Globus Compute to deploy agents on Polaris from this notebook. 

Globus Compute is a managed computing service. In addition to providing a simple Python/REST API for sending tasks to remote computers, it also provides a ``fire-and-forget'' computing model in which a managed cloud service takes responsibility for orchestrating execution of tasks on a remote endpoint. Users can retrieve results after completiion via the cloud service.

This tutorial is configured to use the Globus Compute endpoint hosted by ALCF on the Polaris computer. You can also set up your own endpoint on resources to which you have access by following the Globus Compute documentation. 

Configuration
We first create a Globus Compute executor that we will use to submit and manage tasks. The executor provides a simple asynchronous Python interface.

Note: the first time you create the executor you will need to authenticate with Globus Auth. To use the Polaris endpoint you must authenticate using an ALCF identity.

Globus Compute Multi User Endpoints spawn a unique endpoint process for each user and subsequently allow task exection. The user endpoint is configured according to the user_endpoint_config argument. To use Polaris you must specify: 1) our account and 2) the queue to use. You can optionally specify a config_key to configure the execution environment. In this case, we will load a shared UV environment with various packages preinstalled. If you want to install additional tools, you will need to create a custom environment.


In [ ]:
from globus_compute_sdk import Executor
from globus_compute_sdk.serialize import ComputeSerializer, AllCodeStrategies

POLARIS_EP = "9a947ba5-f537-4681-acf3-cc66485aadec"

gce = Executor(endpoint_id=POLARIS_EP,
               user_endpoint_config={"account": "ATPESC2026", 
                                     "queue": "ATPESC",
                                     'worker_init': 'source /home/yadunand/setup_atpesc_2026.sh'})

To test the endpoint, we can submit a simple hello world function. 

In [ ]:
def hello():
    import academy
    return "Hello from Polaris"

future = gce.submit(hello)
print(future.result())

Now we can extend to deploy a persistent agent via Globus Compute. The agent will stay running as long as the manager context. We can continue to send messages to our running agent via its action interface. Note: we don't include an LLM here for simplicity, but the agent could include a local or remote LLM and make arbitrary local tool calls. 

In [ ]:
from __future__ import annotations

import asyncio

from academy.agent import action
from academy.agent import Agent
from academy.exchange.cloud.client import HttpExchangeFactory
from academy.handle import Handle
from academy.manager import Manager
from globus_compute_sdk import Executor as GCExecutor


class HelloWorldRemote(Agent):
    @action
    async def hello(self, value) -> str:
        import platform
        return f'Hello {value} (from {platform.platform()})'


async def main() -> int:

    async with await Manager.from_exchange_factory(
        factory=HttpExchangeFactory(),
        executors=gce,
    ) as manager:
        agent_handle = await manager.launch(HelloWorldRemote)

        print(await agent_handle.hello('World'))

    return 0


# In a Jupyter notebook, we use top-level `await` to run the async main() coroutine
# because the notebook already has an event loop running.
await main()


In [ ]:
gce.shutdown()